# Attack Zoo, Colab runner

La o sesiune noua se refac pasii 1 si 2.



## 1. Setup


In [ ]:
!git clone https://github.com/Maxxtra/mlsp-attack-zoo.git 2>/dev/null || git -C /content/mlsp-attack-zoo pull
%cd /content/mlsp-attack-zoo
!pip install -q -r requirements.txt
!python src/data.py --download imagenette


## 2. Google Drive

Copiez in Drive rezultatele dupa fiecare model.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Verificarea atacurilor scrise de mana

Comparam FGSM si PGD din `src/my_attacks.py` fata de torchattacks la 1/255, 2/255 si 4/255.
Rulam pe CPU: pe GPU aceleasi convolutii dau rezultate usor diferite intre rulari,
iar acolo unde gradientul e aproape zero semnul se inverseaza.
Trebuie sa iasa `all match`.


In [ ]:
%cd /content/mlsp-attack-zoo
!python src/verify_my_attacks.py --model resnet50 --n 4


## 4. Testul scripturilor pentru server

Varianta mica a grilei L2: un model, 3 atacuri, 2 bugete, n=20.
FGSM si BIM in L2 sunt scrise de mana in `src/my_attacks.py`, torchattacks le are doar in Linf.

In [ ]:
%cd /content/mlsp-attack-zoo
!bash scripts/run_grila_l2_test.sh

Rularile de test au n=20 si nu au ce cauta in tabel. Le sterg dupa ce am vazut ca merg.

In [ ]:
%cd /content/mlsp-attack-zoo
!head -n -6 results/results.csv > /tmp/r.csv && mv /tmp/r.csv results/results.csv
!tail -3 results/results.csv

## 5. Testul pe 1000 de clase

Grila taie cele 1000 de logit-uri ImageNet la cele 10 clase Imagenette, deci o predictie
mutata pe alta clasa ImageNet trece tot ca fiind corecta. Aici evaluez aceleasi imagini
atacate in ambele feluri. Daca pe 1000 acuratetea e mult mai mica, platoul FGSM
e in parte un artefact de masurare.


In [ ]:
%cd /content/mlsp-attack-zoo
!python src/test_1000_clase.py --model resnet50
!python src/test_1000_clase.py --model convnext

## 6. Grila Linf

Per model: fgsm, bim si pgd la 6 bugete, plus deepfool o singura data.
DeepFool cauta perturbatia minima care schimba clasa, deci ignora bugetul.

Daca sesiunea
cade, rulam din nou celulele 1 si 2, aducem inapoi rezultatele cu celula 7 si reluam
aici doar modelele care lipsesc.


In [ ]:
%cd /content/mlsp-attack-zoo
for model in ['resnet50', 'efficientnet', 'convnext', 'vit']:
    print(f"\n===== {model} =====")
    !bash scripts/grid.sh {model}
    !cp results/results.csv /content/drive/MyDrive/results.csv


## 7. Ce e gata si ce lipseste

Numara cate bugete are fiecare pereche model-atac: 6 pentru fgsm, bim si pgd, 1 pentru deepfool.
A doua linie aduce inapoi din Drive rezultatele, dupa o sesiune pierduta.


In [ ]:
%cd /content/mlsp-attack-zoo
!tail -n +2 results/results.csv | cut -d, -f2,3,4 | sort | uniq -c
# !cp /content/drive/MyDrive/results.csv results/results.csv


## 8. Figura

Se genereaza din `results/results.csv`, filtrate pe norma si pe numarul de imagini.


In [ ]:
%cd /content/mlsp-attack-zoo
!python src/make_figures.py --norm linf --n 200
!python src/make_figures.py --norm l2 --n 200
from IPython.display import Image, display
display(Image('figures/robust_acc_imagenette_linf_n200.png'))


## 9. Descarcare


In [ ]:
from google.colab import files
files.download('results/results.csv')
files.download('figures/robust_acc_imagenette_linf_n200.png')